In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib
import os


In [2]:
route_dataset_df = pd.read_pickle(
    "../data/processed/route_dataset.pkl"
)

print(route_dataset_df.shape)

(19647, 15)


In [3]:
day_mapping = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6
}

route_dataset_df["Day of Week"] = (
    route_dataset_df["Day of Week"]
    .map(day_mapping)
)

In [4]:
feature_columns = [
    "Driver ID",
    "Country",
    "Week ID",
    "Day of Week",
    "Planned Stop Count",
    "Planned Distance"
]

X = route_dataset_df[
    feature_columns
]

y = route_dataset_df[
    "RQS"
]

print(X.head())
print()
print(y.head())

   Driver ID  Country  Week ID  Day of Week  Planned Stop Count  \
0          0        1        0            0                   7   
1          1        1        0            0                   7   
2          2        1        0            0                   7   
3          3        1        0            0                  10   
4          4        1        0            0                   8   

   Planned Distance  
0         49.468094  
1         33.274342  
2         12.124804  
3         19.039848  
4         20.632674  

0    0.5000
1    0.5000
2    0.6667
3    0.8400
4    0.6875
Name: RQS, dtype: float64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(15717, 6)
(3930, 6)


In [6]:
rf_regressor = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_regressor.fit(
    X_train,
    y_train
)

print("Regression model trained!")

Regression model trained!


In [7]:
y_pred = rf_regressor.predict(
    X_test
)

print(y_pred[:10])

[0.862022 0.802969 0.980878 0.885463 0.914216 0.635101 0.518012 0.2831
 0.679403 0.832252]


In [11]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

import numpy as np

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    y_pred
)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

MAE  : 0.1044
RMSE : 0.1442
R²   : 0.4225


In [12]:
importance_df = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": rf_regressor.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df)

              Feature  Importance
0           Driver ID    0.303596
5    Planned Distance    0.264564
2             Week ID    0.146802
4  Planned Stop Count    0.128869
3         Day of Week    0.081776
1             Country    0.074393


In [13]:
os.makedirs(
    "../outputs/models",
    exist_ok=True
)

joblib.dump(
    rf_regressor,
    "../outputs/models/random_forest_regressor.pkl"
)

print("Regression model saved successfully!")

Regression model saved successfully!
